# Extreme aerosol event detection (dust, smoke ,etc) from PACE polarimetric L2 products

**POC/Reviewers:** Meng Gao (NASA GSFC 616, SSAI), Kirk Knobelspiesse (NASA GSFC 616)

[edl]: https://urs.earthdata.nasa.gov/
[oci-data-access]: https://oceancolor.gsfc.nasa.gov/resources/docs/tutorials/notebooks/oci_data_access/

## Summary
This notebook demonstrate the detection of high AOD events, and expore their optical and microphysical properties. The results are summarized in a html file. HARP2 FastMAPOL L2 data can be downloaded from either earthdata cloud or ob.daac web.

### load modules

In [1]:
import earthaccess
import requests

import os
import sys
import glob
import shutil
import numpy as np
import xarray as xr

import argparse
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path
from matplotlib import rcParams

#update the path of the tools
mapol_path=os.path.expanduser('~/Documents/GitHub/pace-rapid-response/PRR_OC/dust_copy_4tests/mapoltool')
sys.path.append(mapol_path)

from tools.detection_html_all import *
from tools.detection_plot_map import *
from tools.detection_util import *
from tools.detection_download import *

from matplotlib import rcParams

import earthaccess

### setup system, download l2 data

In [2]:
auth = earthaccess.login(persist=True)
    
# Change default font to something available
rcParams['font.family'] = 'serif' 
rcParams['font.size'] = '12' 

tspan = ("2025-09-15", "2025-09-15")
day1 = tspan[0]+'_'+tspan[1]

data_path, l1c_path, plot_path, html_path = setup_data(tspan)


short_name="PACE_HARP2_L2_MAPOL_OCEAN_NRT"

filelist_l2 = download_l2_cloud(tspan, short_name=short_name)
nfile = len(filelist_l2)
print("total file before selection", nfile)

./data/2025-09-15_2025-09-15 ./data_l1c/2025-09-15_2025-09-15 ./plot/2025-09-15_2025-09-15 ./html/


KeyboardInterrupt: 

### select data

In [ ]:
aod_min = 0.3
npixel_min = 100*100
filev2 = select_data(filelist_l2, aod_min=aod_min, npixel_min=npixel_min)
nfile = len(filev2)
print("total file after selection", nfile)

### make plot test

In [ ]:
file1 = filev2[0]
plot_l1c_l2(filev2[0], plot_path, l1c_path=l1c_path)

### make plot for all selected granules

Run the following code when need download all granules
Note that a large number of plots may be generated. 

```make_plot(filev2, plot_path, l1c_path)```

The function is defined as in detection_util.py

```
def make_plot(filev2, plot_path, l1c_path="./data/", flag_cloud=True):
    """generate plots according to filev2"""
    
    os.makedirs(plot_path, exist_ok=True)
    os.makedirs(l1c_path, exist_ok=True)
    
    for file1 in filev2[:]:
        try:
            plot_l1c_l2(file1, plot_path, l1c_path=l1c_path, flag_cloud=flag_cloud)
        except:
            print('failed', file1)
```

### create html for sharing

In [ ]:
output_file = html_path+"harp2_fastmapol_"+day1+'_n'+str(nfile)+".html"

sequence = [['globe', 'rgb', 'aot', ], ['ssa', 'fvf', 'sph']]
titlev_custom = [["", "", "AOD (550nm)"], ["Single Scattering Albedo (550nm)", 
                                           "Fine Mode Volume Fraction", "Spherical Fraction"]]

#title = 'PACE Rapid Response on HARP2 FastMAPOL L2:\n
#    {} granule found for valid #pixel > {} (when aod(550nm) > {})'.format(nfile, npixel_min, aod_min)

title = "PACE HARP2 FastMAPOL L2 Rapid Response:{}-{}".format(tspan[0], tspan[1])
title2 = "Total {} granule found for valid #pixel > {} (when aod(550nm) > {})".format(nfile, npixel_min, aod_min)
             
print(title)
print(title2)

image_groups = get_images_from_subfolders(plot_path)
create_html_from_subfolders(image_groups, output_file, sequence, title=title, title2=title2,
                             titlev=titlev_custom, resolution_factor=2, quality=75)

### clean up downloaded files, do not add them into repo

share html on:
https://oceancolor.gsfc.nasa.gov/fileshare/meng_gao/rapid_pace